# Insider purchases relative to existing holdings

This notebook accompanies the InsiderAlpha report [A $1M Insider Buy Is Not Always Large](https://insider-alpha.com/blog/2026-08-04-insider-purchase-size-relative-to-holdings). It executes the published event-construction and cohort methodology.

By default it downloads the deterministic 10,000-row public sample. Sample output validates the workflow and schema but **does not reproduce the complete-release estimates**. Set `INSIDERALPHA_DATASET_PATH` to a licensed full-release CSV to reproduce the published study population.

In [ ]:
from __future__ import annotations

import hashlib
import os
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.relative_purchase_size import analyze, load_dataset


## Resolve and verify the input

The bundled public archive contains data and documentation only. The notebook verifies its SHA-256 checksum before reading it. If the sample is removed from a fork, it falls back to the canonical InsiderAlpha download.

In [ ]:
SAMPLE_URL = 'https://insider-alpha.com/downloads/insideralpha-form4-sample-v2.0.0.zip'
CHECKSUM_URL = SAMPLE_URL + '.sha256'
DATA_DIR = ROOT / 'data'
sample_path = DATA_DIR / Path(SAMPLE_URL).name
checksum_path = DATA_DIR / (sample_path.name + '.sha256')

configured_path = os.environ.get('INSIDERALPHA_DATASET_PATH')
if configured_path:
    dataset_path = Path(configured_path).expanduser().resolve()
    input_mode = 'complete release supplied by the researcher'
else:
    if sample_path.exists():
        checksum_line = checksum_path.read_text(encoding='utf-8').strip()
        input_mode = 'bundled 10,000-row public sample'
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(SAMPLE_URL, sample_path)
        checksum_line = urllib.request.urlopen(CHECKSUM_URL).read().decode('utf-8').strip()
        input_mode = 'downloaded 10,000-row public sample'
    expected_sha256 = checksum_line.split()[0]
    actual_sha256 = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    if actual_sha256 != expected_sha256:
        raise ValueError(f'Sample checksum mismatch: {actual_sha256}')
    dataset_path = sample_path

print(f'Input mode: {input_mode}')
print(f'Dataset: {dataset_path}')


## Construct research events

The reusable module filters current non-derivative code `P` rows, requires direct ownership and mature return fields, collapses price tranches and computes `purchased_shares / shares_owned_after`. Fractions above 100% are excluded.

In [ ]:
frame = load_dataset(dataset_path)
result = analyze(frame)

print(f'Source rows: {result.source_rows:,}')
print(f'Eligible aggregated events: {len(result.eligible_events):,}')
display(result.bucket_summary.round(3))


## Compare with the published complete-release results

The table below is loaded from the versioned research package rather than calculated from the public sample. When using the full release, compare the computed cohort counts and medians with this table. Small floating-point display differences are acceptable; event counts must match.

In [ ]:
published = pd.read_csv(ROOT / 'results' / 'published_bucket_results.csv')
display(published)

if configured_path:
    computed_counts = result.bucket_summary.set_index('bucket')['event_count'].astype(int)
    expected_counts = published.set_index('bucket')['event_count'].astype(int)
    comparison = pd.concat(
        [expected_counts.rename('published'), computed_counts.rename('computed')],
        axis=1,
    )
    comparison['matches'] = comparison['published'].eq(comparison['computed'])
    display(comparison)
else:
    print('Sample mode: exact complete-release comparison is intentionally skipped.')


## Visualize the observed ordering

The published ordering is not monotonic. The chart is descriptive and does not represent benchmark-adjusted alpha or an executable trading strategy.

In [ ]:
plot_data = published.set_index('bucket')['median_return_180d_pct']
ax = plot_data.plot(kind='bar', color=['#71717a', '#71717a', '#71717a', '#10b981', '#71717a'])
ax.set_title('Published median 180-day outcome by purchase fraction')
ax.set_xlabel('Purchase as share of reported post-transaction holdings')
ax.set_ylabel('Median 180-day outcome (%)')
ax.tick_params(axis='x', rotation=35)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


## Interpretation limits

- Outcomes begin at the transaction-date market-price baseline, not the first tradable price after EDGAR acceptance.
- Returns are absolute security-price changes, not benchmark- or sector-adjusted alpha.
- Holdings describe a reported security and ownership form, not total personal wealth.
- Code `P` includes private purchases as well as open-market purchases.
- This is an observational event study, not investment advice.